# RNAscope Reanalysis

Batch reanalyse saved RNAscope ROI JSON files from the NWB session folders, save new reanalysis JSON files, and compare the reconstructed counts with the experimenter counts stored in each NWB file.
Depends on JSON files with ROIs drawn by an initial analysis using 'RNAscope analysis' notebook.

## Reanalysis Settings

All paths in this notebook are built from:

`nwb_root = repo / "NWBdata" / "001832"`

After downloading the DANDI dataset, the RNAscope sessions are stored as subject folders directly inside `NWBdata/001832`, for example:

`NWBdata/001832/sub-L1-ST8/`

`NWBdata/001832/sub-L2-ST8/`

`NWBdata/001832/sub-L3-ST6/`

`NWBdata/001832/sub-L4-ST8/`

Each session folder contains the NWB file and the saved ROI analysis JSON files used for reanalysis.

The notebook has two separate switches:

- `run_reanalysis` controls whether the saved ROI analysis JSON files are opened and reanalysed.
- `save_reanalysis` controls whether the newly reanalysed results are written back to disk as new ROI analysis JSON files.

The folder names are controlled here:

`source_analysis_folder = "analysis"`

`reanalysis_folder = "reanalysis"`

With these settings, the notebook reads the original saved ROI analysis JSON files from:

`NWBdata/001832/sub-L*/analysis/*.roi_analysis.json`

If `save_reanalysis = True`, the notebook saves the newly reanalysed ROI analysis JSON files to:

`NWBdata/001832/sub-L*/reanalysis/*.roi_analysis.json`

The settings behave as follows:

`run_reanalysis = True`

`save_reanalysis = True`

opens the original JSON files, reanalyses them, stores the current results in memory as `reanalysis_counts`, and saves new JSON files to the `reanalysis/` folders.

`run_reanalysis = True`

`save_reanalysis = False`

opens the original JSON files, reanalyses them, and stores the current results in memory as `reanalysis_counts`, but does not save new JSON files to disk.

`run_reanalysis = False`

`save_reanalysis = False`

does not rerun the analysis. In this mode, the notebook reloads previously saved reanalysis JSON files from:

`NWBdata/001832/sub-L*/reanalysis/*.roi_analysis.json`

Use this mode only after reanalysis JSON files have already been created.

In short:

- Use `run_reanalysis  = True`  to rerun the analysis.
- Use `save_reanalysis = True`  if you want to write the new reanalysis JSON files to disk.
- Use `run_reanalysis  = False` only when you want to reload existing saved reanalysis JSON files.

In [ ]:
from pathlib import Path
import os
import json
import re
import sys

import pandas as pd
from IPython.display import display, Markdown
from pynwb import NWBHDF5IO

repo = Path(os.environ.get(
    "ANALYSIS_ROOT",
    Path.home() / "Documents" / "Repositories" / "analysis_Belal2026"
))
functions_dir = repo / "Python functions"

if str(functions_dir) not in sys.path:
    sys.path.insert(0, str(functions_dir))

from master_RNAscope import (
    RNAscopeAnalysisFinish,
    reconstruct_state_from_saved_analysis,
    save_rnascope_field_analysis,
    parse_field_metadata,
    counts_from_roi_jsons,
)

nwb_root = repo / "NWBdata" / "001832"

sessions = ["sub-L1-ST8_ses-20240905T115902", 
            "sub-L2-ST8_ses-20240909T100245", 
            "sub-L3-ST6_ses-20240909T112846", 
            "sub-L4-ST8_ses-20240916T101347"]

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

display(Markdown(f"NWB root: `{nwb_root}`"))

In [ ]:
# Reanalysis Settings

run_reanalysis = True
save_reanalysis = False

source_analysis_folder = "analysis"
reanalysis_folder = "reanalysis"
display_mode = "rendered_from_raw"

detection_method = "DoG"
dog_mode = "tolerance"

if detection_method == "DoG":
    new_params = {
        "detection_method": detection_method,
        "dog_mode": dog_mode,
        "sigma_small": 1.0,
        "sigma_large": 2.8,
        "threshold_percentile": 99,
        "peak_footprint": 4,
        "maxima_tolerance": 170,
        "show_detected": False,
        "show_verify": False,
    }
else:
    new_params = {
        "detection_method": detection_method,
        "maxima_tolerance": 160,
        "show_detected": False,
        "show_verify": False,
    }
    
new_params

In [ ]:
# load data

input_rows = []

session_paths = {
    "L1.ST8": {
        "session_root": nwb_root / "sub-L1-ST8",
        "nwb_path": nwb_root / "sub-L1-ST8" / "sub-L1-ST8_ses-20240905T115902.nwb",
    },
    "L2.ST8": {
        "session_root": nwb_root / "sub-L2-ST8",
        "nwb_path": nwb_root / "sub-L2-ST8" / "sub-L2-ST8_ses-20240909T100245.nwb",
    },
    "L3.ST6": {
        "session_root": nwb_root / "sub-L3-ST6",
        "nwb_path": nwb_root / "sub-L3-ST6" / "sub-L3-ST6_ses-20240909T112846.nwb",
    },
    "L4.ST8": {
        "session_root": nwb_root / "sub-L4-ST8",
        "nwb_path": nwb_root / "sub-L4-ST8" / "sub-L4-ST8_ses-20240916T101347.nwb",
    },
}

for session, paths in session_paths.items():
    session_root = paths["session_root"]
    nwb_path = paths["nwb_path"]
    analysis_dir = session_root / source_analysis_folder
    json_paths = sorted(analysis_dir.glob("*.roi_analysis.json"))

    input_rows.append(
        {
            "session": session,
            "nwb_exists": nwb_path.exists(),
            "analysis_dir_exists": analysis_dir.exists(),
            "n_json": len(json_paths),
            "nwb_path": str(nwb_path),
            "analysis_dir": str(analysis_dir),
        }
    )

input_summary = pd.DataFrame(input_rows)
display(input_summary)

missing = input_summary[
    (~input_summary["nwb_exists"])
    | (~input_summary["analysis_dir_exists"])
    | (input_summary["n_json"] == 0)
]

if not missing.empty:
    raise FileNotFoundError("Missing NWB files or analysis JSONs. See input_summary above.")

In [ ]:
# Run Reanalysis
run_rows = []
reanalysis_rows = []

if run_reanalysis:
    for session, paths in session_paths.items():
        session_root = paths["session_root"]
        nwb_path = paths["nwb_path"]
        analysis_dir = session_root / source_analysis_folder
        reanalysis_dir = session_root / reanalysis_folder
        json_paths = sorted(analysis_dir.glob("*.roi_analysis.json"))

        if save_reanalysis:
            reanalysis_dir.mkdir(parents=True, exist_ok=True)

        print(f"{session}: reanalysing {len(json_paths)} fields")

        for json_path in json_paths:
            state, analysis_payload = reconstruct_state_from_saved_analysis(
                nwb_path,
                analysis_path=json_path,
                display_mode=display_mode,
            )

            results, state = RNAscopeAnalysisFinish(state, **new_params)

            field_name = str(state["field"])
            field_meta = parse_field_metadata(field_name)

            for _, row in results.iterrows():
                reanalysis_rows.append(
                    {
                        "condition": field_meta["condition"],
                        "cell_type": str(row["group"]),
                        "slice_id": field_meta["slice_id"],
                        "field": field_name,
                        "hemisphere": field_meta["hemisphere"],
                        "field_index": int(field_meta["field_index"]),
                        "replicate": int(row["roi"]),
                        "count": int(row["count"]),
                        "session": session,
                        "session_group": session,
                        "count_channel": str(state.get("count_channel", "")),
                        "analysis_json": str(json_path),
                    }
                )

            out_path = None
            if save_reanalysis:
                out_path = save_rnascope_field_analysis(
                    state,
                    results,
                    analysis_params=new_params,
                    out_dir=reanalysis_dir,
                )

            run_rows.append(
                {
                    "session": session,
                    "field": state["field"],
                    "count_channel": state.get("count_channel"),
                    "n_roi_rows": len(results),
                    "source_json": str(json_path),
                    "reanalysis_json": str(out_path) if out_path is not None else None,
                }
            )

run_summary = pd.DataFrame(run_rows)
display(run_summary)

reanalysis_counts = pd.DataFrame(
    reanalysis_rows,
    columns=[
        "condition",
        "cell_type",
        "slice_id",
        "field",
        "hemisphere",
        "field_index",
        "replicate",
        "count",
        "session",
        "session_group",
        "count_channel",
        "analysis_json",
    ],
)

display(reanalysis_counts)

In [ ]:
# Build Counts DataFrame From Reanalysis JSONs or Current Run

display_columns = [
    "condition",
    "cell_type",
    "slice_id",
    "field",
    "hemisphere",
    "field_index",
    "replicate",
    "count",
    "session",
    "session_group",
]

if run_reanalysis and not reanalysis_counts.empty:
    display(Markdown("Reanalysis counts built from current in-memory run."))
    display(
        reanalysis_counts[display_columns]
        .sort_values(
            [
                "condition",
                "cell_type",
                "session",
                "hemisphere",
                "field_index",
                "replicate",
            ],
            kind="stable",
        )
        .reset_index(drop=True)
    )

else:
    reanalysis_json_paths = []

    for session, paths in session_paths.items():
        reanalysis_dir = paths["session_root"] / reanalysis_folder
        reanalysis_json_paths.extend(sorted(reanalysis_dir.glob("*.roi_analysis.json")))

    reanalysis_counts = counts_from_roi_jsons(reanalysis_json_paths)

    display(Markdown(f"Reanalysis JSON files: `{len(reanalysis_json_paths)}`"))
    display(
        reanalysis_counts[display_columns]
        .sort_values(
            [
                "condition",
                "cell_type",
                "session",
                "hemisphere",
                "field_index",
                "replicate",
            ],
            kind="stable",
        )
        .reset_index(drop=True)
    )

## Load Experimenter Counts From NWB

In [ ]:
experimenter_dfs = []

for session, paths in session_paths.items():
    nwb_path = paths["nwb_path"]

    with NWBHDF5IO(str(nwb_path), "r", load_namespaces=True) as io:
        nwbfile = io.read()
        df = (
            nwbfile.processing["rnascope_analysis_metadata"]["experimenter_chrnb2_counts"]
            .to_dataframe()
            .reset_index(drop=True)
        )

    df["session"] = df["session"] if "session" in df.columns else session
    df["session_group"] = df["session_group"] if "session_group" in df.columns else session
    experimenter_dfs.append(df)

experimenter_counts = pd.concat(experimenter_dfs, ignore_index=True)
experimenter_counts["field_index"] = experimenter_counts["field_index"].astype(int)
experimenter_counts["replicate"] = experimenter_counts["replicate"].astype(int)
experimenter_counts["count"] = experimenter_counts["count"].astype(int)

display(experimenter_counts)

In [ ]:
# Compare Reanalysis Counts With Experimenter Counts

session_name_map = {
    "sub-L1-ST8": "L1.ST8",
    "sub-L2-ST8": "L2.ST8",
    "sub-L3-ST6": "L3.ST6",
    "sub-L4-ST8": "L4.ST8",
}

reanalysis_counts["session"] = reanalysis_counts["session"].replace(session_name_map)
reanalysis_counts["session_group"] = reanalysis_counts["session_group"].replace(session_name_map)

compare_keys = [
    "condition",
    "cell_type",
    "slice_id",
    "field",
    "hemisphere",
    "field_index",
    "replicate",
    "session",
    "session_group",
]

exact_compare = experimenter_counts[compare_keys + ["count"]].merge(
    reanalysis_counts[compare_keys + ["count", "count_channel"]],
    on=compare_keys,
    how="outer",
    suffixes=("_experimenter", "_reanalysis"),
    indicator=True,
)

exact_compare["count_experimenter"] = exact_compare["count_experimenter"].astype("Int64")
exact_compare["count_reanalysis"] = exact_compare["count_reanalysis"].astype("Int64")
exact_compare["count_delta"] = (exact_compare["count_reanalysis"] - exact_compare["count_experimenter"]).astype("Int64")

display(Markdown("### Exact ROI/replicate comparison"))
display(exact_compare)

display(Markdown("### Merge status"))
display(exact_compare["_merge"].value_counts(dropna=False).rename_axis("merge_status").reset_index(name="n"))

matched = exact_compare[exact_compare["_merge"] == "both"].copy()
if not matched.empty:
    pearson_r = matched["count_experimenter"].corr(matched["count_reanalysis"], method="pearson")
    spearman_r = matched["count_experimenter"].corr(matched["count_reanalysis"], method="spearman")
    print("matched ROI rows:", len(matched))
    print("Pearson r:", round(pearson_r, 4))
    print("Spearman r:", round(spearman_r, 4))
    print("mean delta:", round(matched["count_delta"].mean(), 4))
    print("median delta:", round(matched["count_delta"].median(), 4))

In [ ]:
# Field-Level Mean Comparison

field_keys = [
    "condition",
    "cell_type",
    "slice_id",
    "field",
    "hemisphere",
    "field_index",
    "session",
    "session_group",
]

experimenter_field = (
    experimenter_counts
    .groupby(field_keys, as_index=False)["count"]
    .mean()
    .rename(columns={"count": "experimenter_mean"})
)

reanalysis_field = (
    reanalysis_counts
    .groupby(field_keys, as_index=False)["count"]
    .mean()
    .rename(columns={"count": "reanalysis_mean"})
)

field_compare = experimenter_field.merge(
    reanalysis_field,
    on=field_keys,
    how="outer",
    indicator=True,
)
field_compare["mean_delta"] = field_compare["reanalysis_mean"] - field_compare["experimenter_mean"]

display(field_compare)

matched_field = field_compare[field_compare["_merge"] == "both"].copy()
if not matched_field.empty:
    pearson_r = matched_field["experimenter_mean"].corr(matched_field["reanalysis_mean"], method="pearson")
    spearman_r = matched_field["experimenter_mean"].corr(matched_field["reanalysis_mean"], method="spearman")
    print("matched field rows:", len(matched_field))
    print("Pearson r:", round(pearson_r, 4))
    print("Spearman r:", round(spearman_r, 4))

In [ ]:
# Optional CSV Export
save_csv = False

if save_csv:
    counts_csv = nwb_root / "rnascope_reanalysis_counts.csv"
    exact_compare_csv = nwb_root / "rnascope_reanalysis_vs_experimenter_exact.csv"
    field_compare_csv = nwb_root / "rnascope_reanalysis_vs_experimenter_field_means.csv"

    reanalysis_counts.to_csv(counts_csv, index=False)
    exact_compare.to_csv(exact_compare_csv, index=False)
    field_compare.to_csv(field_compare_csv, index=False)

    print(counts_csv)
    print(exact_compare_csv)
    print(field_compare_csv)